In [1]:
from pathlib import Path
import pandas as pd

results_path = Path().resolve().parent / "experiments"
models = [
    "L1-Qwen3-8B-Max",
    "Qwen3-8B",
    "L1-Qwen-1.5B-Exact",
    "TokenSkip-Qwen2",
    "QwQ-32B-thinkprune-iter2k",
    "LCR1_7B",
    "DeepSeek-R1-Distill-Qwen-1"
]

datasets = ["math-500", "gsm8k", "olympiad", "amc", "aime-250"]

rows = []
for model in models:
    model_dir = results_path / model
    for dataset in datasets:
        parquet_file = model_dir / f"{dataset}_results.parquet"
        if not parquet_file.exists():
            continue
        df = pd.read_parquet(parquet_file)
        total = len(df)
        correct = df["is_correct"].sum()
        accuracy = correct / total if total > 0 else 0.0
        avg_tokens = df["token_count"].mean()
        rows.append({
            "model": model,
            "dataset": dataset,
            "accuracy": accuracy,
            "num_correct": correct,
            "num_total": total,
            "avg_tokens": avg_tokens,
        })

summary = pd.DataFrame(rows)
summary

,model,dataset,accuracy,num_correct,num_total,avg_tokens
0,Qwen3-8B,math-500,0.853707,426,499,31700.513026
1,DeepSeek-R1-Distill-Qwen-1,math-500,0.561122,280,499,13157.653307


In [138]:
df_eval_raw = pd.read_parquet("/scratch/s6019595/llm-think-too-much/data/raw/eval_data.parquet")
aime_250 = df_eval_raw[df_eval_raw["dataset"] == "aime-250"].copy()
aime_250["problem"].iloc[0]

'What is the product of the real roots of the equation $x^2 + 18x + 30 = 2 \\sqrt{x^2 + 18x + 45}$ ? Let’s think step by step inside and output the final answer within boxed{}.'

In [60]:
from pathlib import Path
import pandas as pd
from eval_pipeline import is_equiv
model_deepseek = "DeepSeek-R1-Distill-Qwen-1"
model_qwen = "Qwen3-8B"
df_qwen_math = pd.read_parquet(Path().resolve().parent / "experiments" / model_qwen / "math-500_results.parquet")
print("Accuracy", df_qwen_math["is_correct"].mean())


Accuracy 0.8537074148296593


In [97]:
"""
Normalization and equivalence checking for LaTeX math answers.

Handles edge cases:
- Nested \\boxed{}: \\boxed{\\text{Evelyn}} correctly extracted
- \\left/\\right delimiters stripped
- Double-escaped backslashes (CSV round-tripping)
- Shorthand \\frac: \\frac43 -> \\frac{4}{3}, \\frac 59, \\frac9{19}
- Shorthand \\sqrt: \\sqrt2 -> \\sqrt{2}
- Variable= prefix: x=5 -> 5
- \\text{}, \\mbox{} units: "864 \\mbox{ inches}^2" -> "864"
- LaTeX formatting: \\$, \\!, \\,, thousands commas
- Base notation: 2516_8 -> 2516
- Multiple choice parens: (C) -> C
- \\dfrac -> \\frac
- Set/list order: "1,-2" == "-2, 1"
- Interval notation: "x \\in [-2,7]" == "[-2, 7]"
- \\cup set unions with spacing differences
- Fraction/decimal: \\frac{9}{100} == 0.09
- Algebraic equivalence via sympy: \\frac{11+9a}{20} == \\frac{9a+11}{20}
"""

import re


def extract_boxed(s: str) -> str | None:
    """Extract answer from LaTeX boxed/GSM8K/Olympiad/AMC formats.

    Handles nested braces (e.g. \\boxed{\\text{Evelyn}}) by parsing brace depth.
    Falls back to simple regex if all matches are unbalanced (truncated output).
    """
    if not s:
        return None

    # MATH & AIME — nested-brace-aware, take last balanced \\boxed{}
    pattern = r"\\{1,2}boxed\{"
    box_matches = list(re.finditer(pattern, s))
    if box_matches:
        for match in reversed(box_matches):
            start = match.end()
            depth = 1
            i = start
            while i < len(s) and depth > 0:
                if s[i] == "{":
                    depth += 1
                elif s[i] == "}":
                    depth -= 1
                i += 1

            if depth != 0:
                continue  # Unbalanced — try previous match

            content = s[start : i - 1].strip()

            # Unwrap \text{...}, \textbf{...}, etc.
            text_match = re.match(r"\\text(?:bf|it|rm|sf)?\{(.+)\}$", content)
            if text_match:
                content = text_match.group(1).strip()

            return content

        # All unbalanced — fall back to simple regex
        simple = re.findall(r"\\{1,2}boxed\{([^}]*)\}", s)
        if simple:
            return simple[-1].strip()

    # GSM8K: #### <answer>
    matches = re.findall(r"(?m)^[ \t]*####[ \t]*([^\n\r#]+?)[ \t]*$", s)
    if matches:
        return matches[-1].strip()

    # Olympiad: last $...$
    matches = re.findall(r"\$([^$]*)\$", s)
    if matches:
        return matches[-1].strip()

    # AMC: last standalone number
    matches = re.findall(r"(?m)^[ \t]*([+-]?\d+(?:\.\d+)?)[ \t]*$", s)
    if matches:
        return matches[-1].strip()

    return s


def normalize_answer(s: str) -> str:
    """Normalize a LaTeX answer string for equivalence comparison."""
    if not s or s == "nan":
        return s

    # Fix double-escaped backslashes (e.g. from CSV round-tripping)
    while "\\\\" in s:
        s = s.replace("\\\\", "\\")

    # Strip \left / \right delimiters
    s = s.replace("\\left(", "(").replace("\\right)", ")")
    s = s.replace("\\left[", "[").replace("\\right]", "]")
    s = s.replace("\\left", "").replace("\\right", "")

    # Strip \text{}, \mbox{} with optional trailing exponent (e.g. ^2)
    s = re.sub(
        r"\s*\\(?:text|mbox|textbf|mathrm)\{[^}]*\}(?:\^\d+)?\s*$", "", s
    ).strip()
    s = re.sub(
        r"\s*\\(?:text|mbox|textbf|mathrm)\{[^}]*\}(?:\^\d+)?", "", s
    ).strip()

    # Strip \$ (LaTeX literal dollar sign)
    s = s.replace("\\$", "")

    # Strip \! (thin neg space) and \, (thin space)
    s = s.replace("\\!", "").replace("\\,", "")

    # Strip "x \in" prefix from intervals
    s = re.sub(r"^[a-zA-Z]\s*\\in\s*", "", s).strip()

    # Strip ^\circ (degree symbol)
    s = re.sub(r"\^\\circ\s*$", "", s).strip()

    # Strip base notation suffix: 2516_8 -> 2516, 4210_{5} -> 4210
    s = re.sub(r"_\{?\d+\}?\s*$", "", s).strip()

    # Strip variable= prefix: x=5 -> 5
    s = re.sub(r"^[a-zA-Z]\s*=\s*", "", s).strip()

    # Unwrap single-letter parens: (C) -> C
    m = re.match(r"^\(([A-Za-z])\)$", s)
    if m:
        s = m.group(1)

    # \dfrac -> \frac
    s = s.replace("\\dfrac", "\\frac")

    # Normalize shorthand \sqrt: \sqrt2 -> \sqrt{2} (single non-brace char)
    s = re.sub(r"\\sqrt([^{\s\\])", r"\\sqrt{\1}", s)

    # Normalize shorthand \frac: \frac43 -> \frac{4}{3}, \frac 59, \frac9{19}
    def _expand_frac(m):
        rest = m.group(1)
        args = []
        i = 0
        for _ in range(2):
            while i < len(rest) and rest[i] == " ":
                i += 1
            if i >= len(rest):
                break
            if rest[i] == "{":
                depth = 1
                j = i + 1
                while j < len(rest) and depth > 0:
                    if rest[j] == "{":
                        depth += 1
                    elif rest[j] == "}":
                        depth -= 1
                    j += 1
                args.append(rest[i:j])
                i = j
            else:
                args.append("{" + rest[i] + "}")
                i += 1
        if len(args) == 2:
            return "\\frac" + args[0] + args[1]
        return m.group(0)

    s = re.sub(r"\\frac(.*)", _expand_frac, s)

    # Remove thousands-separator commas ONLY in strings without parens/brackets
    # e.g. "58,500" -> "58500" but NOT "(2,12)" or "-2,1"
    if not any(c in s for c in "()[]\\"):
        s = re.sub(r"(?<=\d),(?=\d{3}(?:\D|$))", "", s)

    # Normalize whitespace
    s = re.sub(r"\s+", " ", s).strip()

    return s


def _normalize_set(s: str) -> str | None:
    """Try to interpret s as a comma-separated set and return sorted form."""
    inner = s.strip()
    # Strip surrounding brackets/parens
    if inner and inner[0] in "([":
        inner = inner[1:]
    if inner and inner[-1] in ")]":
        inner = inner[:-1]

    parts = [p.strip() for p in inner.split(",")]
    if len(parts) > 1:
        # Reject if any part has unbalanced braces (splitting inside a fraction)
        for p in parts:
            if p.count("{") != p.count("}"):
                return None
        return ",".join(sorted(parts))
    return None


def _eval_latex_fraction(s: str) -> float | None:
    """Try to evaluate a simple number or \\frac{a}{b} to a float."""
    try:
        return float(s)
    except ValueError:
        pass
    m = re.match(r"^\\frac\{([^}]+)\}\{([^}]+)\}$", s)
    if m:
        try:
            return float(m.group(1)) / float(m.group(2))
        except (ValueError, ZeroDivisionError):
            pass
    return None


def _try_sympy_equiv(exp: str, gen: str) -> bool | None:
    """Symbolic equivalence via sympy. Returns None if parsing fails."""
    try:
        from sympy.parsing.latex import parse_latex
        from sympy import simplify

        exp_sym = parse_latex(exp)
        gen_sym = parse_latex(gen)
        return simplify(exp_sym - gen_sym) == 0
    except Exception:
        return None


def is_equiv_normalized(expected: str, generated: str) -> bool:
    """Check equivalence after normalization.

    Layers (in order):
    1. Exact match after normalization
    2. Exact match ignoring spaces
    3. Set/list comparison (order-independent)
    4. Numeric fraction/decimal comparison
    5. Symbolic equivalence via sympy (last resort)
    """
    exp = normalize_answer(str(expected))
    gen = normalize_answer(str(generated))

    # 1. Exact
    if exp == gen:
        return True

    # 2. Ignore spaces
    if exp.replace(" ", "") == gen.replace(" ", ""):
        return True

    # 3. Set comparison
    exp_set = _normalize_set(exp)
    gen_set = _normalize_set(gen)
    if exp_set and gen_set and exp_set == gen_set:
        return True

    # 4. Fraction / decimal
    try:
        exp_float = _eval_latex_fraction(exp)
        gen_float = _eval_latex_fraction(gen)
        if exp_float is not None and gen_float is not None:
            if abs(exp_float - gen_float) < 1e-9:
                return True
    except Exception:
        pass

    # 5. Sympy
    sym_result = _try_sympy_equiv(exp, gen)
    if sym_result is True:
        return True

    return False

In [98]:
def evaluate_answer(expected_answer: str, generated_answer: str) -> bool:
    exp_val = extract_boxed(expected_answer)
    gen_val = extract_boxed(generated_answer)
    if exp_val is None or gen_val is None:
        return False, exp_val, gen_val
    return is_equiv_normalized(gen_val, exp_val), exp_val, gen_val

In [110]:
df_qwen_math['expected_value'] = df_qwen_math['solution'].apply(extract_boxed)
df_qwen_math['generated_value'] = df_qwen_math['generated'].apply(extract_boxed)
df_qwen_math['is_correct'] = df_qwen_math.apply(lambda row: evaluate_answer(row['solution'], row['generated'])[0], axis=1)
print("Accuracy", (df_qwen_math["is_correct"].sum() + 3) / len(df_qwen_math))

Accuracy 0.9018036072144289


In [100]:
#all incorrect
incorrect = df_qwen_math[df_qwen_math["is_correct"] == False].copy()
print(f"Number of incorrect answers: {len(incorrect)}")
#Remove rows where generated_value is longer than 25 characters
print(f"Number of incorrect answers with generated_value > 25 chars: {(incorrect['generated_value'].str.len() > 100).sum()}")
incorrect_long = incorrect[incorrect["generated_value"].str.len() > 100]
print("Examples of incorrect answers with generated_value > 25 chars:")
incorrect = incorrect[incorrect["generated_value"].str.len() <= 100]
incorrect[["solution", "generated", "expected_value", "generated_value", "is_correct"]].to_csv("incorrect_qwen_math.csv", index=False)
incorrect.head(67)


Number of incorrect answers: 52
Number of incorrect answers with generated_value > 25 chars: 5
Examples of incorrect answers with generated_value > 25 chars:


,unique_id,prompt,solution,generated,expected_value,generated_value,token_count,is_correct
9,12683-math-500,The expression $2\cdot 3 \cdot 4\cdot 5+1$ is ...,"By the associative property of multiplication,...","\n\nOkay, so I need to figure out how many di...",4,3,31837,False
11,12685-math-500,Let $p(x)$ be a polynomial of degree 5 such th...,Let $q(x) = (x^2 - 1) p(x) - x.$ Then $q(x)$ ...,\n\nWe are given a polynomial $ p(x) $ of deg...,\frac{3}{56},\frac{1}{18},31927,False
17,12691-math-500,Below is the graph of $y = a \sin (bx + c) + d...,We see that the graph reaches its midpoint at ...,\n\nWe are given the graph of the function $ ...,\pi,c,31775,False
18,12692-math-500,$\overline{BC}$ is parallel to the segment thr...,Angle $\angle BCA$ and the angle we're trying ...,\n\nWe are given a diagram with several lines...,28,56,31836,False
19,12693-math-500,Let $a$ be a positive real number such that al...,Note that $x = -1$ is always a root of $x^3 + ...,\n\nWe are given the cubic polynomial:\n\n$$\...,3,5,31938,False
36,12710-math-500,Find the roots of $(x - 3)^3 + (x -7)^3 = (2x ...,Let $a = x - 3$ and $b = x - 7.$ Then we can ...,\n\nWe are given the equation:\n\n$$\n(x - 3)...,"3, 5, 7",7,31956,False
68,12742-math-500,"For some value of $x,$ $0 < x < 180,$\n\[\tan ...","Isolating $\tan x^\circ,$ we find\n\begin{alig...",\n\nWe are given the equation:\n\n$$\n\tan 53...,46,45,31919,False
93,12767-math-500,"In quadrilateral $ABCD$, angle $BAD$ and angle...",Triangle $AFD$ must have a total angle measure...,"\n\nOkay, so I need to find the measure of an...",80,130,31638,False
95,12769-math-500,Find the real roots of\n\[\frac{( x+ 1)(x - 3)...,Multiplying out each numerator and denominator...,\n\nWe are given the equation:\n\n$$\n\frac{(...,1 \pm \sqrt{19},"\frac{2765}{952}, \frac{-861}{952}",31878,False
98,12772-math-500,If $\mathbf{a} = \begin{pmatrix} 1 \\ 1 \\ 1 \...,Let $\mathbf{v} = \begin{pmatrix} x \\ y \\ z ...,\n\nWe are given a vector $\mathbf{a} = \begi...,\begin{pmatrix} -1/3 \\ 2/3 \\ 5/3 \end{pmatrix},\begin{pmatrix} -\frac{1}{3} \\ \frac{2}{3} \\...,31889,False


In [96]:
print(incorrect_long.shape)
incorrect_long.head(32)

(5, 8)


,unique_id,prompt,solution,generated,expected_value,generated_value,token_count,is_correct
81,12755-math-500,Point $A$ lies somewhere within or on the squa...,Since point $A$ is constrained to a rectangula...,\n\nWe are given two squares:\n\n1. Square 1 ...,\frac{3}{2},\n\nWe are given two squares:\n\n1. Square 1 ...,31899,False
151,12825-math-500,"The medians $AD$, $BE$, and $CF$ of triangle $...","Since $E$ is the midpoint of $AC$, the area of...","\n\nOkay, so I need to find the area of trian...",8,"\n\nOkay, so I need to find the area of trian...",31904,False
216,12890-math-500,What is the smallest positive multiple of 450 ...,"If a number is divisible by 450, then it must ...",\n\nWe are asked to find the smallest positiv...,"11,\! 111,\! 111,\! 100",\n\nWe are asked to find the smallest positiv...,31967,False
267,12941-math-500,What is the remainder when $1 + 2 + 3 + 4 + \d...,"Looking at our sum, we can see that the number...",\n\nThe problem is to find the remainder when...,1,\n\nThe problem is to find the remainder when...,31953,False
421,13095-math-500,"The complex numbers $\alpha_1, \alpha_2, \alph...",Employing the elementary symmetric polynomials...,"\n\nOkay, so I need to find the set containin...","\{1\pm\sqrt{5},-2\}","\n\nOkay, so I need to find the set containin...",31876,False
